In [1]:
import cell2mol
import os
from cell2mol.elementdata import ElementData
# from cell2mol.read_write import writexyz
elemdatabase = ElementData()

In [2]:
infopath = "YOBCUO/YOBCUO.info"

In [3]:
from cell2mol.read_write import readinfo
from cell2mol.classes import cell
from cell2mol.cell_reconstruction import classify_fragments, fragments_reconstruct
from cell2mol.charge_assignment import *

In [4]:
name = "YOBCUO"

In [5]:
debug=0

In [6]:
print(f"INITIATING cell object from input") 

# Reads reference molecules from info file, as well as labels and coordinates
labels, pos, ref_labels, ref_fracs, cellvec, cellparam = readinfo(infopath)

# Initiates cell
newcell = cell(name, labels, pos, cellvec, cellparam)
newcell.get_reference_molecules(ref_labels, ref_fracs, debug=debug) 
newcell.assess_errors()

INITIATING cell object from input
MOLECULE.SPLIT COMPLEX: labels=['Ni', 'Br', 'S', 'N', 'N', 'O', 'O', 'O', 'C', 'C', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'C', 'H', 'C', 'H', 'H', 'H', 'C', 'H', 'H', 'H', 'C', 'H', 'C', 'H', 'H', 'H', 'C', 'H', 'H', 'H', 'C', 'H', 'C', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'C', 'C', 'C', 'H', 'C', 'C', 'H', 'C', 'C', 'H', 'H', 'H', 'C', 'H', 'H', 'H', 'C', 'H', 'H', 'H', 'C', 'H', 'H', 'C', 'H', 'H', 'C', 'H', 'H', 'C', 'H', 'H']
MOLECULE.SPLIT COMPLEX: metal_idx=[0]
MOLECULE.SPLIT COMPLEX: rest_idx=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80]
SPLIT COMPLEX: rest labels: ['Br', 'S', 'N', 'N', 'O', 'O', 'O', 'C', 'C', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'C', 'H', 'C', 'H', 'H', 'H', 'C',

In [7]:
newcell.check_missing_H(debug=debug)                                     

False

In [8]:
newcell.reconstruct(debug=2)

CLASSIFY_FRAGMENTS. 24 Blocks sorted as (Molec, Frag, H): 0 8 16

##############################################
FRAG_RECONSTRUCT. 8 molecules submitted to SEQUENTIAL with Heavy
##############################################
goodlist=[]
avglist=[]
badlist=[------------- Cell2mol MOLECULE Object --------------
 Version                      = 0.1
 Type                         = specie
 Sub-Type                     = Heavy
 Number of Atoms              = 8
 Formula                      = H2-C6
 Has Adjacency Matrix         = YES
 Origin                       = cell.get_moleclist
---------------------------------------------------
, ------------- Cell2mol MOLECULE Object --------------
 Version                      = 0.1
 Type                         = specie
 Sub-Type                     = Heavy
 Number of Atoms              = 8
 Formula                      = H2-C6
 Has Adjacency Matrix         = YES
 Origin                       = cell.get_moleclist
-------------------------------------

[------------- Cell2mol MOLECULE Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type                     = molecule
  Number of Atoms              = 81
  Formula                      = H41-C32-N2-O3-S-Ni-Br
  Has Adjacency Matrix         = YES
  Origin                       = cell.reconstruct
  Number of Ligands            = 3
  Number of Metals             = 1
 ---------------------------------------------------,
 ------------- Cell2mol MOLECULE Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type                     = molecule
  Number of Atoms              = 81
  Formula                      = H41-C32-N2-O3-S-Ni-Br
  Has Adjacency Matrix         = YES
  Origin                       = cell.reconstruct
  Number of Ligands            = 3
  Number of Metals             = 1
 ---------------------------------------------------]

In [ ]:
initial_fragments = newcell.get_moleclist().copy()

In [ ]:
molecules, fragments, hydrogens = classify_fragments(initial_fragments, newcell.refmoleclist, debug=debug)

In [ ]:
molecules

In [ ]:
fragments

In [ ]:
hydrogens

In [ ]:
if len(fragments) > 0 or len(hydrogens) > 0: newcell.is_fragmented = True
else:                                        newcell.is_fragmented = False

In [ ]:
newcell.is_fragmented 

In [ ]:
newcell.refmoleclist[0].eleccount

In [ ]:
reconstructed_molecules, Warning = fragments_reconstruct(molecules, 
                                                         fragments, 
                                                         hydrogens, 
                                                         newcell.refmoleclist, 
                                                         newcell.cellvec, 
                                                         newcell.refmoleclist[0].cov_factor, 
                                                         newcell.refmoleclist[0].metal_factor,
                                                        debug = 1)

In [ ]:
adj_types1 = np.load("YOBCUO/adj_types1.npy")
adj_types2 = np.load("YOBCUO/adj_types2.npy")

In [ ]:
adj_types2.shape

In [ ]:
elems = elemdatabase.elementnr.keys()

In [ ]:
print("elem1 - elem2 : reordered - reference")
for kdx, (elem, row1) in enumerate(zip(elems, adj_types1)):
    for ldx, (elem2, val1) in enumerate(zip(elems, row1)):
        val2 = adj_types2[kdx, ldx]
        if val1 != val2: 
            print(f"{kdx} {ldx} {elem} - {elem2} : {val1} - {val2}")

In [ ]:
newcell.refmoleclist[0].metals[0]

In [ ]:
for atom in newcell.refmoleclist[0].metals[0].get_coord_sphere():
    print(atom.label, atom.parents_index)

In [ ]:
elem_mol1=np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0])

In [ ]:
elem_mol2 = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0])

In [ ]:
for idx, (elem, count1, count2) in enumerate(zip(elems, elem_mol1, elem_mol2)):
    if count1 != 0 or count2 != 0:
        print(f"{idx} {elem} {count1} {count2}")

In [ ]:
ref = newcell.refmoleclist[0]

In [ ]:
ref.set_adj_types()

In [ ]:
ref.adj_types.shape

In [ ]:
newcell.refmoleclist[0].adj_types[10]

In [ ]:
reconstructed_molecules

In [ ]:
Warning

In [ ]:
Warning

In [ ]:
for fra in fragments:
    print(fra.coord, fra.frac_coord)

In [ ]:
newcell.reconstruct(debug=debug) 

In [ ]:
newcell.refmoleclist[0]

In [9]:
newcell.error_reconstruction

False

In [10]:
newcell.is_fragmented

False

In [11]:
newcell.refmoleclist

[------------- Cell2mol MOLECULE Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type                     = molecule
  Number of Atoms              = 81
  Formula                      = H41-C32-N2-O3-S-Ni-Br
  Has Adjacency Matrix         = YES
  Number of Ligands            = 3
  Number of Metals             = 1
 ---------------------------------------------------]

In [12]:
newcell.moleclist

[------------- Cell2mol MOLECULE Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type                     = molecule
  Number of Atoms              = 81
  Formula                      = H41-C32-N2-O3-S-Ni-Br
  Has Adjacency Matrix         = YES
  Origin                       = cell.reconstruct
  Number of Ligands            = 3
  Number of Metals             = 1
 ---------------------------------------------------,
 ------------- Cell2mol MOLECULE Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type                     = molecule
  Number of Atoms              = 81
  Formula                      = H41-C32-N2-O3-S-Ni-Br
  Has Adjacency Matrix         = YES
  Origin                       = cell.reconstruct
  Number of Ligands            = 3
  Number of Metals             = 1
 ---------------------------------------------------]

In [13]:
newcell.get_unique_species(debug=debug)

[------------- Cell2mol LIGAND Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type                     = ligand
  Number of Atoms              = 66
  Formula                      = H33-C28-N2-O2-S
  Has Adjacency Matrix         = YES
  Origin                       = split_complex
 ---------------------------------------------------,
 ------------- Cell2mol LIGAND Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type                     = ligand
  Number of Atoms              = 13
  Formula                      = H8-C4-O
  Has Adjacency Matrix         = YES
  Origin                       = split_complex
 ---------------------------------------------------,
 ------------- Cell2mol LIGAND Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type                     = ligand
  Number of Atoms              = 1
  Formula        

In [14]:
selected_cs = []
for idx, spec in enumerate(newcell.unique_species):
#     print("doing", spec)
    if spec.subtype != "metal":
        spec.get_protonation_states()
        spec.get_possible_cs()
        print(spec.formula)
        selected_cs.append(list([cs.corr_total_charge for cs in spec.possible_cs]))
        print(f"{spec.protonation_states=}")
#         for prot in spec.protonation_states :
#              for sub_spec, sub_prot in zip(spec.coord, prot.coords):
#                 print(sub_spec, sub_prot, (sub_spec == sub_prot)) 
#         print("==========")     
#         for cs in spec.possible_cs :
#              for sub_spec, sub_cs_prot in zip(spec.coord, cs.protonation.coords):
#                 print(sub_spec, sub_cs_prot, (sub_spec == sub_cs_prot))     
                
#         print("==========")     
#         for prot, cs in  zip(spec.protonation_states, spec.possible_cs) :
#              for sub_prot, sub_cs_prot in zip(prot.coords, cs.protonation.coords):
#                 print(sub_prot, sub_cs_prot, (sub_prot == sub_cs_prot))   
    else :
        spec.get_possible_cs()
        selected_cs.append(spec.possible_cs)   
    print("")
print(selected_cs)

LIGAND.SPLIT_LIGAND: self.indices=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65]
LIGAND.SPLIT_LIGAND: conn_idx=[0, 1, 2, 3]
LIGAND.SPLIT_LIGAND: conn_labels=['S', 'N', 'N', 'O']
LIGAND.SPLIT_LIGAND: conn_coord=[[7.2868827, 12.0818685, 17.4520631], [7.058379, 12.0657861, 15.8518407], [7.1794209, 10.5617186, 13.434309], [8.0392042, 10.8283577, 17.6128452]]
LIGAND.SPLIT_LIGAND: conn_radii=[1.05, 0.71, 0.71, 0.66]
blocklist=[[0, 1, 3], [2]]
LIGAND.SPLIT_LIGAND: block=[0, 1, 3]
LIGAND.SPLIT_LIGAND: block=[2]
ADD_ATOM: Metalist length 1
ADD_ATOM: Ligand Atoms 66
ADD_ATOM: site= 0
ADD_ATOM: evaluating apos=array([ 7.2868827, 12.0818685, 17.4520631]) and tgt.coord=[7.5712858, 10.2174988, 15.3948914]
ADD_ATOM: received tmpconnec[posadded]=2
ADD_ATOM: Chosen Metal index None. H was added at 

In [15]:
selected_cs

[[-1], [0], [-1], [2, 3]]

In [16]:
final_charge_distribution = balance_charge(newcell.unique_indices, newcell.unique_species, debug=2)

BALANCE: iterlist [[-1], [0], [-1], [2, 3]]
BALANCE: unique_indices [0, 1, 2, 3, 1, 0, 2, 3]
BALANCE: tmpdistr [(-1, 0, -1, 2), (-1, 0, -1, 3)]
BALANCE: alldistr added: [-1, 0, -1, 2, 0, -1, -1, 2]
d=[-1, 0, -1, 2, 0, -1, -1, 2]
BALANCE: alldistr added: [-1, 0, -1, 3, 0, -1, -1, 3]
d=[-1, 0, -1, 2, 0, -1, -1, 2]
d=[-1, 0, -1, 3, 0, -1, -1, 3]


In [17]:
final_charge_distribution

[[-1, 0, -1, 2, 0, -1, -1, 2]]

In [ ]:
def set_charges_create_bonds (specie, unique_indices, unique_species, final_charge_distribution):
        
    spec = unique_species[specie.unique_index]
    indices = [index for index, value in enumerate(unique_indices) if value == specie.unique_index]
    target_charge = [final_charge_distribution[i] for i in indices][0] 
    if debug > 1: print(spec, indices, target_charge)
    
    if (specie.subtype == "molecule" and specie.iscomplex == False) or (specie.subtype == "ligand"):
        formula = specie.formula
        charge_list = [cs.corr_total_charge for cs in spec.possible_cs]
    
    elif specie.subtype == "metal":
        formula = specie.label
        charge_list = spec.possible_cs
    
    if target_charge in charge_list:           
        if debug > 1: print(f"Target charge {target_charge} of {formula} exists in {charge_list}." )
    else:
        if debug > 1: print(f"ERROR: Target charge {target_charge} of {formula} does not exist in {charge_list}." )
        return None
        
    if (specie.subtype == "molecule" and specie.iscomplex == False) or (specie.subtype == "ligand"):
        print(specie.formula)
        specie.get_protonation_states(debug=0)
        specie.get_possible_cs(debug=0)
        formula = specie.formula
        charge_list = [cs.corr_total_charge for cs in specie.possible_cs]        
        
        if target_charge in charge_list:
            if debug > 1: print(f"Target charge {target_charge} of {formula} exists in {charge_list}.")
            idx = charge_list.index(target_charge)
            cs = specie.possible_cs[idx]
            prot = cs.protonation
            specie.set_charges(cs.corr_total_charge, cs.corr_atom_charges, cs.smiles, cs.rdkit_obj)
            specie.create_bonds(debug=0)
        else:
            if debug > 1: print(f"ERROR: Target charge {target_charge} of {formula} does not exist in {charge_list}." )
            return None
                    
    elif specie.subtype == "metal":
        print(specie.label)
        specie.get_possible_cs(debug=0)
        formula = specie.label
        charge_list = spec.possible_cs        

        if target_charge in charge_list:
            if debug > 1: print(f"Target charge {target_charge} of {formula} exists in {charge_list}." )
            idx = charge_list.index(target_charge)
            cs = specie.possible_cs[idx]
            specie.set_charge(cs)         
        else:
            if debug > 1: print(f"ERROR: Target charge {target_charge} of {formula} does not exist in {charge_list}." )
            return None  

In [ ]:
def prepare_mols_v4 (moleclist: list, unique_indices: list, unique_species: list, 
                      selected_cs: list, final_charge_distribution: list, debug: int=0):
    count = 0 
    for mol in moleclist:
        if mol.iscomplex == False:
            set_charges_create_bonds(mol, unique_indices, unique_species, final_charge_distribution)
            count += 1
        
        elif mol.iscomplex:
            tmp_atcharge = np.zeros((mol.natoms))
            tmp_smiles = []
            
            for lig in mol.ligands:            
                set_charges_create_bonds(lig, unique_indices, unique_species, final_charge_distribution)
                count += 1
                 
                tmp_smiles.append(lig.smiles)
                parent_indices = lig.get_parent_indices("molecule")
                for kdx, a in enumerate(parent_indices):
                    tmp_atcharge[a] = lig.atomic_charges[kdx]
                    
            for met in mol.metals:        
                set_charges_create_bonds(met, unique_indices, unique_species, final_charge_distribution)
                count += 1
                parent_index = met.get_parent_index("molecule")
                tmp_atcharge[parent_index] = met.charge     
                
            mol.set_charges(int(sum(tmp_atcharge)), atomic_charges=tmp_atcharge, smiles=tmp_smiles)

    if count != len(final_charge_distribution):
        Warning = True
    else:
        Warning = False
    
    return moleclist, Warning
                
                
#                     ref_data, target_data = arrange_data_for_reorder(spec, mol)
#                     print(f"{spec.labels=}")
#                     print(f"{mol.labels=}")
#                     print(ref_data, target_data)
#                     dummy1, dummy2, map12 = reorder(ref_data, target_data, spec.coord, mol.coord)
                    
#                     print("*****Before function reorder within class****")
#                     print(cs.protonation)
#                     prot = cs.protonation.reorder(map12, debug=debug)
#                     print("*****After function reorder within class****")
#                     print(cs.protonation)
#                     print(prot)
#                     ref_data, target_data = arrange_data_for_reorder(mol, spec)
#                     print(f"{spec.labels=}")
#                     print(f"{mol.labels=}")
#                     print(ref_data, target_data)
#                     dummy1, dummy2, map12 = reorder(ref_data, target_data, mol.coord, spec.coord)
#                     reordered_prot_spec = reorder_protonation(spec.possible_cs[0].protonation, map12, debug=debug)                    
#                     print("**************** After reorder **************** mol vs reordered_protonation")    
#                     for l_mol, c_mol, l_prot, c_prot in zip(mol.labels, mol.coord, prot.labels, prot.coords):
#                         print(l_mol, l_prot, (l_mol == l_prot), c_mol, c_prot, (c_mol==c_prot) )   
#                     print("**************** After reorder **************** mol vs reordered_protonation_spec") 
#                     for l_mol, c_mol, l_prot, c_prot in zip(mol.labels, mol.coord, reordered_prot_spec.labels, reordered_prot_spec.coords):
#                         print(l_mol, l_prot, (l_mol == l_prot), c_mol, c_prot, (c_mol==c_prot) )  

In [ ]:
newcell.moleclist, newcell.error_prepare_mols = prepare_mols_v4(newcell.moleclist, 
                                                                newcell.unique_indices, 
                                                                newcell.unique_species, 
                                                                selected_cs, 
                                                                final_charge_distribution[0], 
                                                                debug=debug)

In [18]:
newcell.assign_charges(debug=debug)

smiles='[H][C+]([N-][c-]1[c+](C([H])(C([H])([H])[H])C([H])([H])[H])[c-]([H])[c+]([H])[c+]([H])[c-]1C([H])(C([H])([H])[H])C([H])([H])[H])[C+]1[C-]([H])[C-]([H])[C+]([H])[C-]([H])[C-]1[N-][S+2]([O-])([O-])[C+]1[C+](C([H])([H])[H])[C-]([H])[C+](C([H])([H])[H])[C-]([H])[C+]1C([H])([H])[H]'
smiles='[H]C(=Nc1c(C([H])(C([H])([H])[H])C([H])([H])[H])c([H])c([H])c([H])c1C([H])(C([H])([H])[H])C([H])([H])[H])c1c([H])c([H])c([H])c([H])c1N=S(=O)([O-])c1c(C([H])([H])[H])c([H])c(C([H])([H])[H])c([H])c1C([H])([H])[H]'
smiles='[H]C(=Nc1c(C([H])(C([H])([H])[H])C([H])([H])[H])c([H])c([H])c([H])c1C([H])(C([H])([H])[H])C([H])([H])[H])c1c([H])c([H])c([H])c([H])c1[N-][S+3](=O)([O-])c1c(C([H])([H])[H])c([H])c(C([H])([H])[H])c([H])c1C([H])([H])[H]'
smiles='[H][C+]([N-][c-]1[c+](C([H])(C([H])([H])[H])C([H])([H])[H])[c-]([H])[c+]([H])[c+]([H])[c-]1C([H])(C([H])([H])[H])C([H])([H])[H])[C+]1[C-]([H])[C-]([H])[C+]([H])[C-]([H])[C-]1[N-][S+2]([O-])([O-])[c-]1[c+](C([H])([H])[H])[c-]([H])[c+](C([H])([H])[H])[c-]([H])[

[------------- Cell2mol MOLECULE Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type                     = molecule
  Number of Atoms              = 81
  Formula                      = H41-C32-N2-O3-S-Ni-Br
  Has Adjacency Matrix         = YES
  Total Charge                 = 0
  Smiles                       = ['[H]C(=Nc1c(C([H])(C([H])([H])[H])C([H])([H])[H])c([H])c([H])c([H])c1C([H])(C([H])([H])[H])C([H])([H])[H])c1c([H])c([H])c([H])c([H])c1N=S(=O)([O-])c1c(C([H])([H])[H])c([H])c(C([H])([H])[H])c([H])c1C([H])([H])[H]', '[H]C1([H])OC([H])([H])C([H])([H])C1([H])[H]', '[Br-]']
  Origin                       = cell.reconstruct
  Number of Ligands            = 3
  Number of Metals             = 1
 ---------------------------------------------------,
 ------------- Cell2mol MOLECULE Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type                     = molecule
  Number of

In [19]:
newcell.error_prepare_mols

False

In [20]:
newcell.moleclist

[------------- Cell2mol MOLECULE Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type                     = molecule
  Number of Atoms              = 81
  Formula                      = H41-C32-N2-O3-S-Ni-Br
  Has Adjacency Matrix         = YES
  Total Charge                 = 0
  Smiles                       = ['[H]C(=Nc1c(C([H])(C([H])([H])[H])C([H])([H])[H])c([H])c([H])c([H])c1C([H])(C([H])([H])[H])C([H])([H])[H])c1c([H])c([H])c([H])c([H])c1N=S(=O)([O-])c1c(C([H])([H])[H])c([H])c(C([H])([H])[H])c([H])c1C([H])([H])[H]', '[H]C1([H])OC([H])([H])C([H])([H])C1([H])[H]', '[Br-]']
  Origin                       = cell.reconstruct
  Number of Ligands            = 3
  Number of Metals             = 1
 ---------------------------------------------------,
 ------------- Cell2mol MOLECULE Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type                     = molecule
  Number of

In [21]:
newcell.assign_spin(debug=2)

------------- Cell2mol GROUP Object --------------
 Version                      = 0.1
 Type                         = specie
 Sub-Type                     = group
 Number of Atoms              = 2
 Formula                      = N-O
 Has Adjacency Matrix         = YES
 Origin                       = split_ligand
 Number of Metals             = 1
---------------------------------------------------

['Ni', 'N', 'O'] [[7.5712858, 10.2174988, 15.3948914], [7.058379, 12.0657861, 15.8518407], [8.0392042, 10.8283577, 17.6128452]]
------------- Cell2mol GROUP Object --------------
 Version                      = 0.1
 Type                         = specie
 Sub-Type                     = group
 Number of Atoms              = 0
 Formula                      = 
 Has Adjacency Matrix         = YES
 Origin                       = split_ligand
 Number of Metals             = 1
---------------------------------------------------

['Ni'] [[7.5712858, 10.2174988, 15.3948914]]
------------- Cell2mol GRO

[------------- Cell2mol MOLECULE Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type                     = molecule
  Number of Atoms              = 81
  Formula                      = H41-C32-N2-O3-S-Ni-Br
  Has Adjacency Matrix         = YES
  Total Charge                 = 0
  Spin                         = 3
  Smiles                       = ['[H]C(=Nc1c(C([H])(C([H])([H])[H])C([H])([H])[H])c([H])c([H])c([H])c1C([H])(C([H])([H])[H])C([H])([H])[H])c1c([H])c([H])c([H])c([H])c1N=S(=O)([O-])c1c(C([H])([H])[H])c([H])c(C([H])([H])[H])c([H])c1C([H])([H])[H]', '[H]C1([H])OC([H])([H])C([H])([H])C1([H])[H]', '[Br-]']
  Origin                       = cell.reconstruct
  Number of Ligands            = 3
  Number of Metals             = 1
 ---------------------------------------------------,
 ------------- Cell2mol MOLECULE Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type        

In [22]:
newcell.error_prepare_mols

False

In [23]:
for mol in newcell.moleclist:
    if mol.iscomplex:
        for met in mol.metals:
            print(met.label, met.spin)

Ni 3
Ni 3


In [24]:
newcell.save(f"{name}/{name}.cell")

SAVING cell2mol CELL object to YOBCUO/YOBCUO.cell
